SMS SPAM DETECTION
BINARY CLASSFICATION

**IMPORTING MODULES**

In [1]:
import numpy as np 
import pandas as pd 
import tensorflow as tf
import re
from nltk.corpus import stopwords

import matplotlib.pyplot as plt
from keras.models import *
from keras.models import load_model
from keras.callbacks import EarlyStopping,ReduceLROnPlateau,ModelCheckpoint

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout, LSTM, Bidirectional

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SMSSpamCollection.csv', error_bad_lines=False)
df.head()

,class,text,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:

df = df.rename(columns={'class': 'type', 'text': 'message'})
df.head()


,type,message,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN


**FEATURE ENGINEERING AND DATA CLEANING**

In [5]:
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    #  lowercase
    text = text.lower()
    #  special characters
    text = re.sub(r'[^0-9a-zA-Z]', ' ', text)
    # extra spaces
    text = re.sub(r'\s+', ' ', text)
    # stopwords
    text = " ".join(word for word in text.split() if word not in STOPWORDS)
    return text

# clean the messages
df['clean_text'] = df['message'].apply(clean_text)
df.head()

,type,message,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,nah think goes usf lives around though


In [6]:
df.type.value_counts()
#The model is more inclined to predict more ham than spam

ham     4827
spam     747
Name: type, dtype: int64

In [7]:
df['type'] = df['type'].map( {'spam': 1, 'ham': 0} )
df.head()

,type,message,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,clean_text
0,0,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,go jurong point crazy available bugis n great ...
1,0,Ok lar... Joking wif u oni...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,free entry 2 wkly comp win fa cup final tkts 2...
3,0,U dun say so early hor... U c already then say...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u dun say early hor u c already say
4,0,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,nah think goes usf lives around though


In [8]:
df= pd.DataFrame(df, columns = ['type','message','clean_text'])

df

,type,message,clean_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goes usf lives around though
...,...,...,...
5569,1,This is the 2nd time we have tried 2 contact u...,2nd time tried 2 contact u u 750 pound prize 2...
5570,0,Will ü b going to esplanade fr home?,b going esplanade fr home
5571,0,"Pity, * was in mood for that. So...any other s...",pity mood suggestions
5572,0,The guy did some bitching but I acted like i'd...,guy bitching acted like interested buying some...


**MODEL BUILDING, TRAINING AND TESTING**

In [9]:

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

X = df['message'].values
y = df['type'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [10]:

token = Tokenizer()
token.fit_on_texts(X_train)
enc_train = token.texts_to_sequences(X_train)
enc_test = token.texts_to_sequences(X_test)
vocab_size = len(token.word_index) + 1
print(enc_train[0:2])

[[2565, 478, 1011, 2566, 66, 574, 15, 1980, 81, 574, 2, 2567, 94, 170, 53, 57, 178, 300, 4, 1981, 2568, 41, 123, 663, 137, 2569, 269, 204, 2570, 809, 300], [1982, 421, 34, 13, 752, 197, 2, 2571]]


In [11]:

max_length = 8
padding_type = "post"
trunc_type = "post" 
padded_train = pad_sequences(enc_train, maxlen=max_length, padding='post', truncating=trunc_type)
padded_test = pad_sequences(enc_test, maxlen=max_length, padding='post', truncating=trunc_type)
print(padded_train)

[[2565  478 1011 ...  574   15 1980]
 [1982  421   34 ...  197    2 2571]
 [ 173   21   30 ...  502 1265 3797]
 ...
 [ 121  402 7978 ...    4  413   19]
 [  24  111   36 ...   65    4   55]
 [ 129   72   38 ...    9 3175 1161]]


In [22]:

model = Sequential()
model.add(Embedding(vocab_size, 24, input_length=max_length))
model.add(Flatten())
model.add(Dense(500, activation='relu'))
model.add(Dense(200, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(100, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(model.summary())

Model: "sequential_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_2 (Embedding)      (None, 8, 24)             191592    
_________________________________________________________________
flatten_2 (Flatten)          (None, 192)               0         
_________________________________________________________________
dense_8 (Dense)              (None, 500)               96500     
_________________________________________________________________
dense_9 (Dense)              (None, 200)               100200    
_________________________________________________________________
dropout_2 (Dropout)          (None, 200)               0         
_________________________________________________________________
dense_10 (Dense)             (None, 100)               20100     
_________________________________________________________________
dense_11 (Dense)             (None, 1)                

In [23]:
loss, accuracy = model.evaluate(padded_test, y_test, verbose=0)
print('Accuracy: %f' % (accuracy*100))

Accuracy: 80.807173


In [14]:
preds = (model.predict(padded_test) > 0.5).astype("int32")

In [15]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

def c_report(y_true, y_pred):
    print("Classification Report")
    print(classification_report(y_true, y_pred))
    acc_sc = accuracy_score(y_true, y_pred)
    print("Accuracy : "+ str(acc_sc))
    return acc_sc


**METRIC BEING TRACKED**
ACCURACY-The number of True Positives and True negatives that were classified correctly
PRECISION -The number of True positives that were correctly classified as positives
F1 SCORE-Harmonic mean of the model’s precision and recall.


In [16]:
c_report(y_test, preds)

Classification Report
              precision    recall  f1-score   support

           0       0.84      0.02      0.04       954
           1       0.14      0.98      0.25       161

    accuracy                           0.16      1115
   macro avg       0.49      0.50      0.15      1115
weighted avg       0.74      0.16      0.07      1115

Accuracy : 0.15964125560538117


0.15964125560538117

**TUNING THE MODEL**

In [17]:
early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)
model.fit(x=padded_train, 
          y=y_train, 
          epochs=50,
          validation_data=(padded_test, y_test), verbose=1,
          callbacks=[early_stop]
          )


Epoch 1/50
140/140 [==============================] - 2s 9ms/step - loss: 0.2727 - accuracy: 0.8874 - val_loss: 0.1469 - val_accuracy: 0.9641
Epoch 2/50
140/140 [==============================] - 1s 8ms/step - loss: 0.0515 - accuracy: 0.9843 - val_loss: 0.1199 - val_accuracy: 0.9650
Epoch 3/50
140/140 [==============================] - 1s 9ms/step - loss: 0.0083 - accuracy: 0.9980 - val_loss: 0.1432 - val_accuracy: 0.9650
Epoch 4/50
140/140 [==============================] - 1s 9ms/step - loss: 0.0019 - accuracy: 0.9996 - val_loss: 0.1846 - val_accuracy: 0.9695
Epoch 5/50
140/140 [==============================] - 1s 9ms/step - loss: 1.3180e-04 - accuracy: 1.0000 - val_loss: 0.2087 - val_accuracy: 0.9695
Epoch 6/50
140/140 [==============================] - 1s 9ms/step - loss: 5.9862e-05 - accuracy: 1.0000 - val_loss: 0.2251 - val_accuracy: 0.9695
Epoch 7/50
140/140 [==============================] - 1s 9ms/step - loss: 4.4715e-05 - accuracy: 1.0000 - val_loss: 0.2368 - val_accuracy: 0

In [18]:
es=EarlyStopping(monitor='val_loss',mode='min',verbose=1,patience=3)
mc=ModelCheckpoint('best_model.h5',monitor='val_acc',mode='max',save_best_only=True,verbose=1)
reduce_learningrate = ReduceLROnPlateau(monitor='val_loss',factor=0.2,patience=3,verbose=1)

callbacks_list = [es,mc,reduce_learningrate]

In [19]:
model.fit(x=padded_train, 
          y=y_train, 
          epochs=50,
          validation_data=(padded_test, y_test), verbose=1,
          callbacks=callbacks_list
          )

Epoch 1/50
140/140 [==============================] - 1s 8ms/step - loss: 1.4594e-05 - accuracy: 1.0000 - val_loss: 0.2821 - val_accuracy: 0.9668
Epoch 2/50
140/140 [==============================] - 1s 8ms/step - loss: 1.2768e-05 - accuracy: 1.0000 - val_loss: 0.2872 - val_accuracy: 0.9668
Epoch 3/50
140/140 [==============================] - 1s 8ms/step - loss: 8.1387e-06 - accuracy: 1.0000 - val_loss: 0.2919 - val_accuracy: 0.9668
Epoch 4/50
140/140 [==============================] - 1s 9ms/step - loss: 7.2781e-06 - accuracy: 1.0000 - val_loss: 0.2960 - val_accuracy: 0.9668

Epoch 00004: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
Epoch 00004: early stopping
